# Indicator 1: Permutation Entropy

**Core idea:** Rank the last `m` prices from lowest to highest. The rank sequence
is an *ordinal fingerprint*. Count how often each fingerprint appears across a
rolling window. Shannon entropy over those frequencies measures how disordered
(chaotic) recent price action is.

| PE value | Market state | Signal |
|----------|-------------|--------|
| Low (< threshold) | Few dominant patterns → trending | 1 (proceed with trade) |
| High (≥ threshold) | All patterns equally likely → chaotic | 0 (step aside) |

`threshold = 0.75` — allows trading during the ~40% most orderly market conditions.
Previously 0.70, which filtered out valid trend periods.

**Why momentum mode matters:** PE is used as a *gate*, not a directional signal.
It only controls whether the three directional indicators (MACD, Fibonacci, MA Crossover)
are allowed to generate a trade.

In [ ]:
import math
import numpy as np

In [ ]:
# ---- Indicator 1: Permutation Entropy (Chaos / Order Filter) ---------------
#
# threshold=0.75: allows trading in the ~40% most orderly market conditions.

def _permutation_entropy_score(prices, m=3, normalize=True):
    """Compute Permutation Entropy for a price list.

    Parameters
    ----------
    prices    : list[float]  recent price window
    m         : int          embedding dimension — ordinal pattern length (default 3)
    normalize : bool         divide by log(m!) to map result to [0, 1]

    Returns
    -------
    float in [0, 1]:  0 = fully ordered,  1 = fully chaotic
    """
    n = len(prices)
    if n < m:
        return 1.0  # insufficient data: treat as chaotic

    pattern_counts = {}
    for i in range(n - m + 1):
        pattern = tuple(int(x) for x in np.argsort(prices[i:i + m]))
        pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1

    total   = sum(pattern_counts.values())
    entropy = 0.0
    for count in pattern_counts.values():
        p = count / total
        entropy -= p * math.log(p)

    if normalize:
        max_h   = math.log(math.factorial(m))
        entropy = entropy / max_h if max_h > 0 else 0.0

    return entropy


def indicator_1_pe(prices_series, date_idx, window=30, m=3, threshold=0.75):
    """Permutation Entropy regime filter.

    Parameters
    ----------
    prices_series : list[float]  full price history
    date_idx      : int          current bar index (0-based)
    window        : int          lookback bars for entropy calculation (default 30)
    m             : int          embedding dimension (default 3)
    threshold     : float        entropy cutoff — below = orderly (default 0.75)

    Returns
    -------
    1  market is orderly  (PE < threshold) — proceed with directional signals
    0  market is chaotic  (PE >= threshold) — step aside
    """
    if date_idx < window + m - 1:
        return 0
    recent = prices_series[date_idx - window + 1:date_idx + 1]
    pe     = _permutation_entropy_score(recent, m=m)
    return 1 if pe < threshold else 0